# Study 930 — When-Issued Window — the teardown

Three windows around a US spin-off, on one hand-compiled table of **26** liquid
US events (distributions 2012-05-01 → 2025-02-24, ≈2/yr,
52 tickers):

1. **parent**, close of the session *after* the announcement → last close *before* the
   child's first regular-way session (never crossing the distribution, so Yahoo's
   inconsistent spin-off adjustment never enters);
2. **child**, first **5 / 10 / 21** regular-way sessions, entered at the *first*
   regular-way close so the day-one print is excluded;
3. **parent + child**, ratio-weighted, 21 sessions from that same close.

Every leg is `asset − SPY` over identical calendar sessions: a cash-neutral spread, so
excess-of-cash *is* the spread. Frictions: **10 bps one-way × NAV** on the spin-off side
(round trip = 2×), 1 bp on SPY, **40 bp/yr borrow on the short index leg**. Total-return
closes (`auto_adjust=True`). One execution lag, everywhere.

*Real-tape numbers below are frozen from [`docs/results.md`](../docs/results.md)
(Fingerprint `e78725a81ebe`, as-of 2026-06-30). Live cells run the **synthetic** control only
and say so.*

In [1]:
R = {'asof': '2026-06-30', 'fp': 'e78725a81ebe', 'n_events': 26, 'n_tickers': 52, 'first': '2012-05-01', 'last': '2025-02-24', 'per_year': 2.0, 'n_delisted': 4, 'par_mean': -1.45, 'par_med': -2.28, 'par_hit': 0.42, 'par_t': -0.26, 'par_hac': -0.47, 'par_ci_lo': -11.43, 'par_ci_hi': 9.47, 'c5_mean': -4.54, 'c5_med': -4.36, 'c5_hit': 0.31, 'c5_t': -2.73, 'c5_hac': -3.02, 'c5_ci_lo': -7.67, 'c5_ci_hi': -1.42, 'c5_hit_lo': 0.17, 'c5_hit_hi': 0.5, 'c5_sd': 8.47, 'c10_mean': -3.47, 'c10_med': -3.09, 'c10_hit': 0.31, 'c10_t': -1.61, 'c10_hac': -2.2, 'c10_ci_lo': -7.61, 'c10_ci_hi': 0.63, 'c21_mean': -1.02, 'c21_med': -1.87, 'c21_hit': 0.46, 'c21_t': -0.41, 'c21_hac': -0.58, 'c21_ci_lo': -5.75, 'c21_ci_hi': 3.73, 'comb_mean': 1.88, 'comb_med': 2.8, 'comb_hit': 0.62, 'comb_t': 1.28, 'comb_hac': 1.44, 'comb_ci_lo': -1.02, 'comb_ci_hi': 4.69, 'n_par': -2.1, 'n_par_t': -0.38, 'n_c5': -4.77, 'n_c5_t': -2.87, 'n_c5_hac': -3.17, 'n_c10': -3.71, 'n_c10_t': -1.71, 'n_c21': -1.27, 'n_c21_t': -0.51, 'n_comb': 1.62, 'n_comb_t': 1.1, 'fw_legs': 5, 'fw_p_single': 0.0088, 'fw_p_fwe': 0.0452, 'era_e_n': 6, 'era_e_mean': -3.77, 'era_e_t': -2.91, 'era_l_n': 20, 'era_l_mean': -4.77, 'era_l_t': -2.23, 'iwm_mean': -4.41, 'iwm_t': -2.48, 'mdy_mean': -4.64, 'mdy_t': -2.62, 'jk_lo': -3.36, 'jk_hi': -2.44, 'dec_n': 19, 'dec_mean': -5.09, 'dec_t': -2.88, 'cost0': -4.56, 'cost0_t': -2.74, 'cost25': -5.07, 'cost25_t': -3.05, 'cost50': -5.57, 'cost50_t': -3.35, 'anchor_exact': 25, 'anchor_tail': 24, 'anchor_tail_lo': 3, 'anchor_tail_hi': 15, 'rw_m3': -6.85, 'rw_m3_t': -2.74, 'rw_m1': -7.76, 'rw_m1_t': -3.78, 'rw_p1': -3.31, 'rw_p1_t': -1.76, 'rw_p3': 2.24, 'rw_p3_t': 1.62, 'ann_m5': -3.78, 'ann_p5': -0.89, 'ratio_half': 1.75, 'ratio_two': 1.41, 'ratio_eq': 0.37, 'caar_m3': 4.09, 'caar_m1': 1.86, 'caar_p1': -1.08, 'caar_p2': -3.9, 'caar_p3': -6.09, 'caar_p5': -4.48, 'caar_p10': -3.49, 'caar_p21': -0.79, 'caar_n': 24, 'h3_mean': -5.91, 'h3_t': -3.65, 'h3_hac': -4.49, 'h3_hit': 0.12, 'h63_mean': -0.74, 'h63_t': -0.19, 'sh0': 4.32, 'sh0_t': 2.6, 'sh0_hit': 0.65, 'sh0_ci_lo': 1.2, 'sh0_ci_hi': 7.45, 'sh25': 3.82, 'sh25_t': 2.3, 'sh25_ci_lo': 0.7, 'sh25_ci_hi': 6.95, 'sh50': 3.33, 'sh50_t': 2.0, 'sh50_hit': 0.62, 'sh50_ci_lo': 0.2, 'sh50_ci_hi': 6.46, 'sh100': 2.33, 'sh100_t': 1.4, 'sh100_ci_lo': -0.79, 'sh100_ci_hi': 5.46, 'sl5_days': 118, 'sl5_of': 3228, 'sl5_bps': -75.9, 'sl5_t': -1.84, 'sl5_lo': -133.6, 'sl5_hi': -9.8, 'sl21_days': 453, 'sl21_of': 3244, 'sl21_bps': -4.2, 'sl21_t': -0.27, 'syn_planted_child': 6.63, 'syn_planted_child_t': 4.43, 'syn_planted_par': 21.72, 'syn_planted_par_t': 2.6, 'syn_null_fire': 0}

## 1. The three legs

Headline figures are **gross**. The child's alpha is negative, so charging a long position's round-trip costs to it would make the effect look bigger than it is; the net column is printed beside it to show frictions are not what create it, and the costed number that matters lives in §5, on the side you would actually trade.

> 💡 **In plain words:** five numbers were pre-specified. Four are noise. The one that fires points the opposite way to the story it was built to test — and once you pay for having taken five looks, it clears the 5% line by a whisker, not a mile.

In [2]:
print(f"{'leg':34s}{'mean':>9s}{'med':>9s}{'hit':>7s}{'t':>8s}{'HAC t':>8s}   95% CI")
rows = [('parent  ann -> distribution', R['par_mean'], R['par_med'], R['par_hit'], R['par_t'], R['par_hac'], R['par_ci_lo'], R['par_ci_hi']),
        ('child   first  5 sessions',  R['c5_mean'],  R['c5_med'],  R['c5_hit'],  R['c5_t'],  R['c5_hac'],  R['c5_ci_lo'],  R['c5_ci_hi']),
        ('child   first 10 sessions',  R['c10_mean'], R['c10_med'], R['c10_hit'], R['c10_t'], R['c10_hac'], R['c10_ci_lo'], R['c10_ci_hi']),
        ('child   first 21 sessions',  R['c21_mean'], R['c21_med'], R['c21_hit'], R['c21_t'], R['c21_hac'], R['c21_ci_lo'], R['c21_ci_hi']),
        ('parent+child first 21',      R['comb_mean'],R['comb_med'],R['comb_hit'],R['comb_t'],R['comb_hac'],R['comb_ci_lo'],R['comb_ci_hi'])]
for lbl, m, md_, h, t, hac, lo, hi in rows:
    print(f"{lbl:34s}{m:+8.2f}%{md_:+8.2f}%{h:7.2f}{t:+8.2f}{hac:+8.2f}   [{lo:+.2f}, {hi:+.2f}]%")
print(f"\nnet of a LONG position's frictions: parent {R['n_par']:+.2f}% "
      f"(t={R['n_par_t']:+.2f}) | child+5d {R['n_c5']:+.2f}% (t={R['n_c5_t']:+.2f}, "
      f"HAC {R['n_c5_hac']:+.2f}) | +10d {R['n_c10']:+.2f}% (t={R['n_c10_t']:+.2f}) | "
      f"+21d {R['n_c21']:+.2f}% (t={R['n_c21_t']:+.2f}) | comb {R['n_comb']:+.2f}% "
      f"(t={R['n_comb_t']:+.2f})")
print('the 5-session leg clears |t|>=2 gross AND net, with the sign inverted vs the claim')
print()
print(f"but five legs were pre-specified, so the family is priced: Westfall-Young")
print(f"max-|t| bootstrap -> single-test p = {R['fw_p_single']:.4f}, "
      f"FAMILY-WISE p over {R['fw_legs']} legs = {R['fw_p_fwe']:.4f}  <- survives, narrowly")

leg                                    mean      med    hit       t   HAC t   95% CI
parent  ann -> distribution          -1.45%   -2.28%   0.42   -0.26   -0.47   [-11.43, +9.47]%
child   first  5 sessions            -4.54%   -4.36%   0.31   -2.73   -3.02   [-7.67, -1.42]%
child   first 10 sessions            -3.47%   -3.09%   0.31   -1.61   -2.20   [-7.61, +0.63]%
child   first 21 sessions            -1.02%   -1.87%   0.46   -0.41   -0.58   [-5.75, +3.73]%
parent+child first 21                +1.88%   +2.80%   0.62   +1.28   +1.44   [-1.02, +4.69]%

net of a LONG position's frictions: parent -2.10% (t=-0.38) | child+5d -4.77% (t=-2.87, HAC -3.17) | +10d -3.71% (t=-1.71) | +21d -1.27% (t=-0.51) | comb +1.62% (t=+1.10)
the 5-session leg clears |t|>=2 gross AND net, with the sign inverted vs the claim

but five legs were pre-specified, so the family is priced: Westfall-Young
max-|t| bootstrap -> single-test p = 0.0088, FAMILY-WISE p over 5 legs = 0.0452  <- survives, narrowly


## 2. Robustness on the 5-session leg

Era cut by distribution date, a size-matched benchmark swap (children are small/mid caps and SPY is not), a leave-one-out jackknife, de-clustering to one event per calendar quarter, and a cost sweep.

> 💡 **In plain words:** costs make a *negative* number more negative, so they cannot be what manufactured it.

In [3]:
print(f"eras : pre-2020 n={R['era_e_n']:2d} {R['era_e_mean']:+.2f}% (t={R['era_e_t']:+.2f})  |  "
      f"2020+ n={R['era_l_n']:2d} {R['era_l_mean']:+.2f}% (t={R['era_l_t']:+.2f})   -> negative in both")
print(f"bench: SPY {R['c5_mean']:+.2f}% (t={R['c5_t']:+.2f})  IWM {R['iwm_mean']:+.2f}% "
      f"(t={R['iwm_t']:+.2f})  MDY {R['mdy_mean']:+.2f}% (t={R['mdy_t']:+.2f})   -> not a size tilt")
print(f"jack : leave-one-out t in [{R['jk_lo']:+.2f}, {R['jk_hi']:+.2f}]   -> no single event carries it")
print(f"clust: one event per quarter n={R['dec_n']} {R['dec_mean']:+.2f}% (t={R['dec_t']:+.2f})")
print(f"cost : 0bps {R['cost0']:+.2f}% (t={R['cost0_t']:+.2f}) | 25bps {R['cost25']:+.2f}% "
      f"(t={R['cost25_t']:+.2f}) | 50bps {R['cost50']:+.2f}% (t={R['cost50_t']:+.2f})")

eras : pre-2020 n= 6 -3.77% (t=-2.91)  |  2020+ n=20 -4.77% (t=-2.23)   -> negative in both
bench: SPY -4.54% (t=-2.73)  IWM -4.41% (t=-2.48)  MDY -4.64% (t=-2.62)   -> not a size tilt
jack : leave-one-out t in [-3.36, -2.44]   -> no single event carries it
clust: one event per quarter n=19 -5.09% (t=-2.88)
cost : 0bps -4.56% (t=-2.74) | 25bps -5.07% (t=-3.05) | 50bps -5.57% (t=-3.35)


## 3. The three non-tape inputs, swept

The event table is an **assumption**: announcement dates, first regular-way sessions and distribution ratios are hand-compiled from Form 10 / 8-K filings and the press. Each is swept.

> 💡 **In plain words:** the regular-way anchor is the one that matters. Start earlier — inside the when-issued tail — and the effect is *stronger*; start three sessions late and it flips sign. That is a description of a three-to-five-session effect, but it also means the dates have to be right to the session.

In [4]:
print('regular-way anchor (the load-bearing proxy):')
for lbl, m, t in [('-3 sessions', R['rw_m3'], R['rw_m3_t']), ('-1 session ', R['rw_m1'], R['rw_m1_t']),
                  (' 0 (table) ', R['n_c5'], R['n_c5_t']), ('+1 session ', R['rw_p1'], R['rw_p1_t']),
                  ('+3 sessions', R['rw_p3'], R['rw_p3_t'])]:
    flag = '   <- sign flips' if m > 0 else ''
    print(f'   {lbl}: {m:+6.2f}% (t={t:+.2f}){flag}')
print(f"announcement date (parent leg): -5 {R['ann_m5']:+.2f}% .. +5 {R['ann_p5']:+.2f}% "
      f"-> null at every setting")
print(f"distribution ratio (combined) : x0.5 {R['ratio_half']:+.2f}% | x1 {R['n_comb']:+.2f}% | "
      f"x2 {R['ratio_two']:+.2f}% | equal-weight {R['ratio_eq']:+.2f}% -> null under every weighting")
print()
print('is the anchor on the right side of the transition? checked, not asserted:')
print(f"  {R['anchor_exact']}/{R['n_events']} stated dates are themselves a session on the"
      ' child tape (VLTO snaps 3 sessions forward: Yahoo has no 2023-10-02/03)')
print(f"  {R['anchor_tail']}/{R['n_events']} children show a genuine when-issued tail of "
      f"{R['anchor_tail_lo']}-{R['anchor_tail_hi']} sessions BEFORE the anchor")
print('  -> the anchor is not set early, which is the direction that would inflate this')

regular-way anchor (the load-bearing proxy):
   -3 sessions:  -6.85% (t=-2.74)
   -1 session :  -7.76% (t=-3.78)
    0 (table) :  -4.77% (t=-2.87)
   +1 session :  -3.31% (t=-1.76)
   +3 sessions:  +2.24% (t=+1.62)   <- sign flips
announcement date (parent leg): -5 -3.78% .. +5 -0.89% -> null at every setting
distribution ratio (combined) : x0.5 +1.75% | x1 +1.62% | x2 +1.41% | equal-weight +0.37% -> null under every weighting

is the anchor on the right side of the transition? checked, not asserted:
  25/26 stated dates are themselves a session on the child tape (VLTO snaps 3 sessions forward: Yahoo has no 2023-10-02/03)
  24/26 children show a genuine when-issued tail of 3-15 sessions BEFORE the anchor
  -> the anchor is not set early, which is the direction that would inflate this


## 4. Event-time shape — EXPLORATORY

CAAR relative to the first regular-way close; negative event time sits **inside the when-issued period**. Only 5 / 10 / 21 were pre-specified — everything else on this grid was searched after seeing the data and carries an unpriced multiple-comparison penalty. It is here for the mechanism, never for the stamp.

In [5]:
for k, v in [(-3, R['caar_m3']), (-1, R['caar_m1']), (0, 0.0), (1, R['caar_p1']),
             (2, R['caar_p2']), (3, R['caar_p3']), (5, R['caar_p5']),
             (10, R['caar_p10']), (21, R['caar_p21'])]:
    bar = '#' * int(abs(v) * 3)
    print(f"  session {k:+3d}: {v:+6.2f}%  {bar}")
print(f"\ntrough at +3 sessions: {R['h3_mean']:+.2f}% (t={R['h3_t']:+.2f}, HAC {R['h3_hac']:+.2f}, "
      f"{R['h3_hit']:.0%} positive); gone by +63d: {R['h63_mean']:+.2f}% (t={R['h63_t']:+.2f})")
print(f"n={R['caar_n']} events have >=3 when-issued sessions on the tape")

  session  -3:  +4.09%  ############
  session  -1:  +1.86%  #####
  session  +0:  +0.00%  
  session  +1:  -1.08%  ###
  session  +2:  -3.90%  ###########
  session  +3:  -6.09%  ##################
  session  +5:  -4.48%  #############
  session +10:  -3.49%  ##########
  session +21:  -0.79%  ##

trough at +3 sessions: -5.91% (t=-3.65, HAC -4.49, 12% positive); gone by +63d: -0.74% (t=-0.19)
n=24 events have >=3 when-issued sessions on the tape


## 5. Harvesting a negative alpha — the short side, and the borrow it needs

The trade is: short the child, long SPY, five sessions. The child's borrow rate is **not on this tape** — it is an assumption, and for a name five days old it is the assumption that decides everything.

> 💡 **In plain words:** at any plausible borrow the arithmetic works; the question is whether a locate exists at all.

In [6]:
print(f"{'child borrow':>14s}{'mean':>9s}{'t':>8s}{'hit':>7s}   95% CI")
rows = [('0%/yr', R['sh0'], R['sh0_t'], R['sh0_hit'], R['sh0_ci_lo'], R['sh0_ci_hi']),
        ('25%/yr', R['sh25'], R['sh25_t'], R['sh0_hit'], R['sh25_ci_lo'], R['sh25_ci_hi']),
        ('50%/yr', R['sh50'], R['sh50_t'], R['sh50_hit'], R['sh50_ci_lo'], R['sh50_ci_hi']),
        ('100%/yr', R['sh100'], R['sh100_t'], R['sh50_hit'], R['sh100_ci_lo'], R['sh100_ci_hi'])]
for lbl, m, t, h, lo, hi in rows:
    flag = '   <- CI through zero' if lo < 0 else ''
    print(f"{lbl:>14s}{m:+8.2f}%{t:+8.2f}{h:7.2f}   [{lo:+.2f}, {hi:+.2f}]%{flag}")
print(f"\ndaily hedged sleeve, 5d window: {R['sl5_days']} active of {R['sl5_of']} "
      f"sessions, {R['sl5_bps']:+.1f} bps/day, HAC t={R['sl5_t']:+.2f}, "
      f"block-boot CI [{R['sl5_lo']:+.1f}, {R['sl5_hi']:+.1f}] bps")
print(f"21d window: {R['sl21_days']} of {R['sl21_of']} sessions, "
      f"{R['sl21_bps']:+.1f} bps/day (t={R['sl21_t']:+.2f}) -- diluted to nothing")
print(f"idle {1 - R['sl5_days']/R['sl5_of']:.0%} of even its OWN window (first distribution"
      ' -> last exit, not whatever pre-history the shared cache holds for SPY):')
print('a capacity statement about 2 events/yr, not evidence either way, which is')
print('why no fund-level Sharpe is quoted')

  child borrow     mean       t    hit   95% CI
         0%/yr   +4.32%   +2.60   0.65   [+1.20, +7.45]%
        25%/yr   +3.82%   +2.30   0.65   [+0.70, +6.95]%
        50%/yr   +3.33%   +2.00   0.62   [+0.20, +6.46]%
       100%/yr   +2.33%   +1.40   0.62   [-0.79, +5.46]%   <- CI through zero

daily hedged sleeve, 5d window: 118 active of 3228 sessions, -75.9 bps/day, HAC t=-1.84, block-boot CI [-133.6, -9.8] bps
21d window: 453 of 3244 sessions, -4.2 bps/day (t=-0.27) -- diluted to nothing
idle 96% of even its OWN window (first distribution -> last exit, not whatever pre-history the shared cache holds for SPY):
a capacity statement about 2 events/yr, not evidence either way, which is
why no fund-level Sharpe is quoted


## 6. Live synthetic control — the estimator is unbiased (synthetic, not the real tape)

A planted world (a real parent run-up and a real child drift) must fire; a null world must not. Same code path as the real tape: `event_panel` on a synthetic frame that uses the loader's schema.

In [7]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from when_issued import data, strategy as st

# SYNTHETIC worlds only — this cell never touches the real tape.
px, ev, truth = data.synthetic_daily(signal_strength=1.0, seed=930)
pl = st.synthetic_detect(px, ev)
print(f"planted (child 21d alpha {truth['planted_child_alpha_21d']:.2%}): "
      f"parent {pl['parent_mean_pct']:+.2f}% (t={pl['parent_t']:+.2f}), "
      f"child21 {pl['child21_mean_pct']:+.2f}% (t={pl['child21_t']:+.2f})")
nulls = [st.synthetic_detect(*data.synthetic_daily(signal_strength=0.0, seed=930+s)[:2])
         for s in range(8)]
pt = np.array([d['parent_t'] for d in nulls]); ct = np.array([d['child21_t'] for d in nulls])
print(f"null x8: parent t mean {pt.mean():+.2f} (|t|>=2 in {(abs(pt)>=2).sum()}/8), "
      f"child21 t mean {ct.mean():+.2f} (|t|>=2 in {(abs(ct)>=2).sum()}/8)")
print('an unbiased ruler -> the real-tape result is a property of spin-offs, not the harness')

planted (child 21d alpha 5.25%): parent +21.72% (t=+2.60), child21 +6.63% (t=+4.43)


null x8: parent t mean +0.16 (|t|>=2 in 0/8), child21 t mean +0.36 (|t|>=2 in 0/8)
an unbiased ruler -> the real-tape result is a property of spin-offs, not the harness


## Verdict

- **Signal — Real, sign inverted, family-wise *p* = 0.0452.**
  -4.54% gross over the child's first five regular-way sessions
  (*t* = -2.73, HAC -3.02, bootstrap CI
  [-7.67, -1.42]% clear of zero, hit rate 0.31
  [0.17, 0.50]; -4.77% net of a long position's
  frictions, which is why gross is the headline). Negative gross and net, in both eras, versus
  SPY / IWM / MDY, under a jackknife (-3.36…-2.44) and after
  de-clustering (-2.88) — |*t*| ≥ 2.2 on every cut. The other four pre-specified
  legs — the parent run-up (-1.45%, *t* = -0.26), the 10- and
  21-session child windows, and the sum-of-the-parts (+1.88%,
  *t* = +1.28) — are null, and the Westfall-Young max-|*t*| bootstrap over all
  five puts the family-wise *p* at **0.0452**: inside 5%, but not comfortably.
  The forced-*seller* prediction is rejected; the shape (pop, three-session slide, month-long
  fill-in) is the index-inclusion footprint of forced *buying*. Survivorship named on this
  axis: 26 curated liquid spins, plus 4 dropped because the child
  was acquired and delisted. Unit beta is assumed throughout — the IWM/MDY swap is the check.
- **Tradability — Fragile.** The harvesting trade is a five-session short of a five-day-old
  listing: +4.32%/trade (*t* = +2.60) gross of borrow,
  +3.33% at 50%/yr, +2.33% with a CI through zero at 100%/yr — and
  the borrow is the one input the tape cannot price. ~2 trades a year,
  8.5 pp per-trade dispersion, and a +2.24% sign flip if the anchor
  is three sessions late. Not bankable as a strategy; useful as a warning about buying the
  first print.